# 🔁 Notebook 3 — Train Siamese Repair Verifier (ResNet18)

**Model:** Siamese CNN (shared ResNet-18 encoder)  
**Output:** `siamese_resnet18_repair.pt`  
**Runtime:** GPU (T4 recommended)

## Strategy
- Build synthetic before/after pairs from pothole road images.
- Labels:
  - `1.0` repaired (inpainted-like variant)
  - `0.5` partially repaired
  - `0.0` not repaired (different damaged patch)
- Train with MSE loss on similarity score.

> This notebook includes strict checks so training won't start if images are missing.

In [ ]:
# ── 1) Check GPU ─────────────────────────────────────────────────────────────
!nvidia-smi
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

In [ ]:
# ── 2) Install dependencies ──────────────────────────────────────────────────
!pip install -q --upgrade kagglehub opencv-python-headless albumentations
print('✅ Dependencies installed')

In [ ]:
# ── 3) Mount Google Drive ────────────────────────────────────────────────────
from google.colab import drive
import os

drive.mount('/content/drive')
SAVE_DIR = '/content/drive/MyDrive/pothole_models'
os.makedirs(SAVE_DIR, exist_ok=True)
print('✅ Save dir:', SAVE_DIR)

In [ ]:
# ── 4) Kaggle authentication ─────────────────────────────────────────────────
import os, json
from getpass import getpass

KAGGLE_USERNAME = input('Kaggle username: ').strip()
KAGGLE_KEY = getpass('Kaggle API key: ').strip()

os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump({'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY}, f)
os.chmod('/root/.kaggle/kaggle.json', 0o600)
os.environ['KAGGLE_USERNAME'] = KAGGLE_USERNAME
os.environ['KAGGLE_KEY'] = KAGGLE_KEY
print('✅ Kaggle auth configured')

In [ ]:
# ── 5) Download source datasets ──────────────────────────────────────────────
import kagglehub
from pathlib import Path

pothrgbd_path = Path(kagglehub.dataset_download('mahyeks/pothrgbd-rgb-and-depth-images-of-potholes'))
annotated_path = Path(kagglehub.dataset_download('chitholian/annotated-potholes-dataset'))

print('✅ PothRGBD:', pothrgbd_path)
print('✅ Annotated:', annotated_path)

In [ ]:
# ── 6) Build image pool ──────────────────────────────────────────────────────
from pathlib import Path
import random

random.seed(42)

def list_images(root: Path):
    out = []
    for ext in ['*.jpg', '*.jpeg', '*.png']:
        out.extend(root.rglob(ext))
    return out

rgbd_imgs = [p for p in list_images(pothrgbd_path) if 'depth' not in str(p).lower()]
ann_imgs = list_images(annotated_path)

all_imgs = []
seen = set()
for p in rgbd_imgs + ann_imgs:
    key = str(p)
    if key not in seen:
        seen.add(key)
        all_imgs.append(p)

print(f'RGBD images: {len(rgbd_imgs)}')
print(f'Annotated images: {len(ann_imgs)}')
print(f'Total unique images: {len(all_imgs)}')

if len(all_imgs) < 200:
    raise RuntimeError('Too few images for Siamese training. Need at least ~200 images.')

# Keep manageable subset for Colab speed
MAX_IMAGES = 6000
if len(all_imgs) > MAX_IMAGES:
    all_imgs = random.sample(all_imgs, MAX_IMAGES)
    print(f'Using subset: {len(all_imgs)} images')

In [ ]:
# ── 7) Pair synthesis helpers ────────────────────────────────────────────────
import cv2
import numpy as np

def load_rgb(path, size=224):
    img = cv2.imread(str(path))
    if img is None:
        return None
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (size, size), interpolation=cv2.INTER_LINEAR)
    return img

def make_repaired(img):
    # inpaint-ish smoothing around random pothole-like region
    h, w, _ = img.shape
    cx, cy = np.random.randint(w//4, 3*w//4), np.random.randint(h//4, 3*h//4)
    rx, ry = np.random.randint(20, 45), np.random.randint(20, 45)
    mask = np.zeros((h, w), dtype=np.uint8)
    cv2.ellipse(mask, (cx, cy), (rx, ry), 0, 0, 360, 255, -1)
    repaired = cv2.inpaint(cv2.cvtColor(img, cv2.COLOR_RGB2BGR), mask, 7, cv2.INPAINT_TELEA)
    repaired = cv2.cvtColor(repaired, cv2.COLOR_BGR2RGB)
    return repaired

def make_partial(img):
    out = img.copy()
    noise = np.random.normal(0, 8, out.shape).astype(np.int16)
    out = np.clip(out.astype(np.int16) + noise, 0, 255).astype(np.uint8)
    out = cv2.GaussianBlur(out, (3, 3), 0)
    return out

In [ ]:
# ── 8) Siamese dataset ───────────────────────────────────────────────────────
import torch
from torch.utils.data import Dataset

class SiameseRepairDataset(Dataset):
    def __init__(self, image_paths, pairs_per_epoch=12000, size=224):
        self.image_paths = image_paths
        self.pairs_per_epoch = pairs_per_epoch
        self.size = size

    def __len__(self):
        return self.pairs_per_epoch

    def _to_tensor(self, img):
        x = img.astype(np.float32) / 255.0
        x = (x - np.array([0.485, 0.456, 0.406], np.float32)) / np.array([0.229, 0.224, 0.225], np.float32)
        x = torch.from_numpy(np.transpose(x, (2, 0, 1))).float()
        return x

    def __getitem__(self, idx):
        p1 = random.choice(self.image_paths)
        img1 = load_rgb(p1, self.size)
        while img1 is None:
            p1 = random.choice(self.image_paths)
            img1 = load_rgb(p1, self.size)

        r = random.random()
        if r < 0.40:
            img2 = make_repaired(img1)
            y = 1.0
        elif r < 0.70:
            img2 = make_partial(img1)
            y = 0.5
        else:
            p2 = random.choice(self.image_paths)
            img2 = load_rgb(p2, self.size)
            while img2 is None:
                p2 = random.choice(self.image_paths)
                img2 = load_rgb(p2, self.size)
            y = 0.0

        return self._to_tensor(img1), self._to_tensor(img2), torch.tensor([y], dtype=torch.float32)

print('✅ Siamese dataset class ready')

In [ ]:
# ── 9) Model definition ──────────────────────────────────────────────────────
import torch.nn as nn
import torchvision.models as models

class SiameseNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        self.encoder = nn.Sequential(*list(backbone.children())[:-1])  # (B,512,1,1)
        self.head = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(256, 1),
            nn.Sigmoid(),
        )

    def embed(self, x):
        z = self.encoder(x)
        return z.flatten(1)

    def forward(self, x1, x2):
        z1 = self.embed(x1)
        z2 = self.embed(x2)
        d = torch.abs(z1 - z2)
        return self.head(d)

print('✅ Siamese model defined')

In [ ]:
# ── 10) DataLoaders + train setup ────────────────────────────────────────────
from torch.utils.data import DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

random.shuffle(all_imgs)
split = int(0.85 * len(all_imgs))
train_imgs = all_imgs[:split]
val_imgs = all_imgs[split:]

train_ds = SiameseRepairDataset(train_imgs, pairs_per_epoch=12000, size=224)
val_ds = SiameseRepairDataset(val_imgs, pairs_per_epoch=2500, size=224)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

model = SiameseNetwork().to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

print('Train batches:', len(train_loader), '| Val batches:', len(val_loader))
print('Device:', device)

In [ ]:
# ── 11) Train Siamese model ──────────────────────────────────────────────────
import time

EPOCHS = 15
best_val = float('inf')
best_path = '/content/siamese_resnet18_repair.pt'

for epoch in range(1, EPOCHS + 1):
    model.train()
    t0 = time.time()
    train_loss = 0.0

    for x1, x2, y in train_loader:
        x1, x2, y = x1.to(device), x2.to(device), y.to(device)
        optimizer.zero_grad(set_to_none=True)
        pred = model(x1, x2)
        loss = criterion(pred, y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for x1, x2, y in val_loader:
            x1, x2, y = x1.to(device), x2.to(device), y.to(device)
            pred = model(x1, x2)
            loss = criterion(pred, y)
            val_loss += loss.item()

    train_loss /= max(1, len(train_loader))
    val_loss /= max(1, len(val_loader))
    dt = time.time() - t0

    print(f'Epoch {epoch:02d}/{EPOCHS} | train={train_loss:.4f} | val={val_loss:.4f} | {dt:.1f}s')

    if val_loss < best_val:
        best_val = val_loss
        torch.save(model.state_dict(), best_path)
        print(f'  ✅ Saved best checkpoint (val={best_val:.4f})')

print('\n✅ Training finished')
print('Best model:', best_path)

In [ ]:
# ── 12) Quick threshold sanity check ─────────────────────────────────────────
# Backend thresholds:
# repaired >= 0.85, partial >= 0.60, else not_repaired

model.load_state_dict(torch.load('/content/siamese_resnet18_repair.pt', map_location=device))
model.eval()

def label_from_score(s):
    if s >= 0.85:
        return 'repaired'
    if s >= 0.60:
        return 'partially_repaired'
    return 'not_repaired'

with torch.no_grad():
    x1, x2, y = next(iter(val_loader))
    x1, x2 = x1.to(device), x2.to(device)
    scores = model(x1, x2).squeeze(1).detach().cpu().numpy()

print('First 10 predictions:')
for i, s in enumerate(scores[:10]):
    print(f'  {i:02d}: score={s:.3f} -> {label_from_score(float(s))}')

In [ ]:
# ── 13) Save to Drive + download ─────────────────────────────────────────────
import shutil
from pathlib import Path
from google.colab import files

best_src = Path('/content/siamese_resnet18_repair.pt')
if not best_src.exists():
    raise FileNotFoundError('Model file missing. Run training cell first.')

best_dst = Path(SAVE_DIR) / 'siamese_resnet18_repair.pt'
shutil.copy(best_src, best_dst)
print('✅ Saved to Drive:', best_dst)

files.download(str(best_src))